# WhiteSearch — EDA: GWOSC Public Strain Data

This notebook demonstrates the full preprocessing pipeline for GWOSC data
and runs a toy injection/recovery test for the black-to-white bounce model.

**Steps:**
1. Load GW150914 strain data (mock if GWPy not available)
2. Preprocess: bandpass, notch, PSD estimation, whitening
3. Simulate a black-to-white bounce signal injection
4. Run toy inference (BilbyRunner with toy sampler)
5. Visualise posterior and summary statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from whitesearch.dataio import GWOSCLoader
from whitesearch.preprocess import GWPreprocessor
from whitesearch.models import BlackToWhiteBounce, StandardBHRingdown
from whitesearch.simulators import GravitationalWaveSimulator
from whitesearch.likelihoods import GWLikelihood
from whitesearch.inference import BilbyRunner, compute_bayes_factor
from whitesearch.validation import InjectionRecovery

plt.rcParams['figure.dpi'] = 120
print('WhiteSearch imports OK')

## 1. Load GWOSC Strain Data

In [ ]:
loader = GWOSCLoader(cache_dir='../artifacts/gwosc')
data = loader.load_event('GW150914', detectors=['H1', 'L1'])

strain_H1 = data['H1']['strain']
times_H1  = data['H1']['times']
sample_rate = data['H1']['sample_rate']
source = data['H1']['source']

print(f'Source: {source}')
print(f'Duration: {len(strain_H1)/sample_rate:.1f} s')
print(f'Sample rate: {sample_rate} Hz')
print(f'Strain RMS: {np.std(strain_H1):.2e}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
t_rel = times_H1 - times_H1[0]
ax.plot(t_rel, strain_H1, 'b', lw=0.3, alpha=0.7)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Strain h(t)')
ax.set_title(f'GW150914 — H1 raw strain ({source})')
plt.tight_layout()
plt.show()

## 2. Preprocess: Bandpass + Notch + PSD + Whitening

In [ ]:
preprocessor = GWPreprocessor(
    sample_rate=sample_rate,
    low_freq=20.0,
    high_freq=1700.0,
)
prep = preprocessor.prepare_raw(strain_H1)

print('Preprocessing quality metrics:')
for k, v in prep['quality'].items():
    print(f'  {k}: {v}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Raw strain
axes[0].plot(t_rel, strain_H1, 'b', lw=0.3)
axes[0].set_ylabel('Raw strain')
axes[0].set_title('H1 strain processing pipeline')

# Whitened strain
axes[1].plot(t_rel, prep['strain_whitened'], 'g', lw=0.3)
axes[1].set_ylabel('Whitened strain')

# PSD
axes[2].loglog(prep['freqs_psd'], np.sqrt(prep['psd']), 'r', lw=1)
axes[2].set_xlabel('Frequency [Hz]')
axes[2].set_ylabel('ASD [Hz^{-1/2}]')
axes[2].set_xlim(10, 2000)

plt.tight_layout()
plt.show()

## 3. Simulate a Black-to-White Bounce Signal Injection

In [ ]:
# Use a 4-second segment for injection
context = {
    'sample_rate': sample_rate,
    'duration': 4.0,
    't_merger': 1.0,
    'low_freq_cutoff': 20.0,
    'rng_seed': 42,
}

true_params = {
    'M': 65.0,
    'a_star': 0.7,
    'log10_tau_bounce_yr': 5.0,
    'log10_ell_q': 3.0,
    'p_lifetime': 4,
    'eps_f': 0.08,
    'eps_Q': -0.15,
    'log10_A_bounce': -21.5,
    'D_L': 400.0,
    'i': 0.5,
    'eta_r': 1e-5,
    'eta_gamma': 1e-6,
}

simulator = GravitationalWaveSimulator()
injected = simulator.simulate(true_params, context, rng=np.random.default_rng(42))

# Check injection summary stats
model = BlackToWhiteBounce()
stats = model.summary_stats(true_params)
print('Injected signal summary:')
for k, v in stats.items():
    print(f'  {k}: {v:.4g}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
t = injected.metadata['times']

# Signal only (noiseless)
signal_only = injected.data - injected.noise_realisation

axes[0].plot(t, signal_only * 1e22, 'b', lw=1, label='Signal')
axes[0].set_ylabel('Strain × 10²²')
axes[0].set_title(f'Injected bounce signal: M={true_params["M"]} M☉, D_L={true_params["D_L"]} Mpc')
axes[0].legend()

# Signal + noise
axes[1].plot(t, injected.data * 1e22, 'r', lw=0.5, alpha=0.8, label='Signal + noise')
axes[1].plot(t, signal_only * 1e22, 'b', lw=1, label='Signal only', alpha=0.7)
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Strain × 10²²')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Run Toy Inference (BilbyRunner)

In [ ]:
# Small run for demonstration (nlive=50, toy sampler if bilby unavailable)
runner = BilbyRunner(
    sampler='dynesty',
    nlive=50,
    outdir='../artifacts/bilby_demo',
    seed=42,
)

ll_bounce = GWLikelihood('bounce')
ll_bh     = GWLikelihood('bh_ringdown')

bh_model = StandardBHRingdown()

print('Running bounce model inference...')
result_bounce = runner.run(ll_bounce, injected, context, model, label='bounce_demo')
print(f'  ln Z (bounce)    = {result_bounce.log_evidence:.3f} ± {result_bounce.log_evidence_err:.3f}')

print('Running BH ringdown inference...')
result_bh = runner.run(ll_bh, injected, context, bh_model, label='bh_demo')
print(f'  ln Z (BH ringdown) = {result_bh.log_evidence:.3f} ± {result_bh.log_evidence_err:.3f}')

In [ ]:
# Compute Bayes factor
bf_result = compute_bayes_factor(
    {'bounce': result_bounce, 'null': result_bh},
    wh_model='bounce',
    null_model='null',
)

print('\n=== Bayes Factor Report ===')
print(f"ln Z (bounce)     = {bf_result['ln_Z_wh']:.3f}")
print(f"ln Z (BH ringdown) = {bf_result['ln_Z_null']:.3f}")
print(f"ln BF (bounce / BH) = {bf_result['ln_BF_vs_null']:.3f} ± {bf_result['ln_BF_vs_null_err']:.3f}")
print(f"Interpretation: {bf_result['interpretation_vs_null']}")
print(f"Internal gate passed: {bf_result['gate_internal_passed']}")

## 5. Posterior Visualisation

In [ ]:
posterior = result_bounce.posterior
params_to_plot = ['M', 'a_star', 'eps_f', 'eps_Q']
params_to_plot = [p for p in params_to_plot if p in posterior.columns]

try:
    import corner
    fig = corner.corner(
        posterior[params_to_plot].values,
        labels=params_to_plot,
        truths=[true_params.get(p, None) for p in params_to_plot],
        truth_color='red',
        show_titles=True,
    )
    plt.suptitle('Bounce model posterior', y=1.02)
    plt.show()
except ImportError:
    # Fallback: simple 1D marginals
    fig, axes = plt.subplots(1, len(params_to_plot), figsize=(4 * len(params_to_plot), 3))
    for ax, p in zip(axes, params_to_plot):
        ax.hist(posterior[p], bins=30, density=True, color='steelblue', alpha=0.7)
        if p in true_params:
            ax.axvline(true_params[p], color='red', ls='--', label='True')
        ax.set_xlabel(p)
        ax.set_ylabel('Density')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 6. Quick Injection / Recovery Check

In [ ]:
ir = InjectionRecovery(
    simulator=simulator,
    runner=runner,
    n_injections=5,
    rng_seed=0,
)
ir_result = ir.run_injections(model, ll_bounce, context)
summary = ir_result.summary()
print('Injection/Recovery Coverage Summary:')
print(summary.to_string(index=False))

## Summary

This notebook demonstrated the end-to-end WhiteSearch pipeline:
- GWOSC data loading and preprocessing (bandpass, PSD, whitening)
- Black-to-white bounce signal injection
- Bayesian model comparison via Bayes factor
- Posterior visualisation and injection/recovery coverage check

**Next steps:**
- Run `whitesearch inject --model bounce --n-injections 100` for a full campaign
- Run `snakemake --snakefile workflows/Snakefile --cores 4 all_gw` for the full pipeline